#Step 0 — Instlling a lybrary for parsing CONLLU files

In [ ]:
!pip install pyconll

#Step 1 — Parsing the Annotation Zip File

In [ ]:
import zipfile
from pathlib import Path
from collections import defaultdict
import os

# Path to your zip file
zip_path = "/content/dump_DigPhil_Diachronic_Treebanks_clean 3.zip"

# Temporary extraction folder
extract_path = Path("extracted_annotations")
extract_path.mkdir(exist_ok=True)

# Open and extract the zip
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

# Check if the path if directory and reassign the right path (the issue appeared after
# manual replacement of the files and zipping the folder on my mac. Mac adds macOS metadata)
for filename in os.listdir(extract_path):
    if os.path.isdir(f"/content/{extract_path}/{filename}"):
        extract_path = Path(f"/content/{extract_path}/{filename}")
        break
# List annotator folders
annotator_folders = [f for f in extract_path.iterdir() if f.is_dir()and f.name != ".ipynb_checkpoints"]
print("Found annotator folders:", [f.name for f in annotator_folders])

# Collect filenames per annotator
annotator_files = {}
for folder in annotator_folders:
    if folder.name != ".ipynb_checkpoints":
        files = [f.name for f in folder.glob("*.conllu")]
        annotator_files[folder.name] = set(files)

annotator_pairs = {}
# Find common files across all annotators
pair = None
ref = []
common_files = {}
for annotator1, files1 in annotator_files.items():
    for annotator2, files2 in annotator_files.items():
        #print(ref)
        if annotator1 != annotator2 and annotator2 not in ref:
            pair = f"{annotator1}_{annotator2}"
            common_files[pair] = set.intersection(files1, files2)
        ref.append(annotator1)
for v,k in common_files.items():
    print(f"{v}:{len(k)}, {k}")




#Step 2 — Hard-coding Samples by Period Dictionary

In [ ]:
# Your raw text as a string
data = """
1750_1799
svediakorp-sec277-EnbomPU_MedborgeligtSkalde.conllu
1750_1799
svediakorp-sec839-bellman15.conllu*
1750_1799
svediakorp-sec268-DulciU_VitterhetsNojen3.conllu
1750_1799
svediakorp-sec267-DulciU_VitterhetsNojen2.conllu*
1700_1749
svediakorp-letter141673-Stalhammar.conllu
1700_1749
svediakorp-sec25-Runius.conllu
1700_1749
svediakorp-sec192-1732-BrennerSE_PoetiskeDikter2.conllu***
1700_1749
svediakorp-sec330-GyllenborgC_SwenskaSpratthoken.conllu
1800_1849
svediakorp-sec991-spf148.conllu
1800_1849
svediakorp-sec252-BremerF_Teckningar1.conllu
1800_1849
svediakorp-sec324-GranbergPA_Enslighetsalskaren.conllu
1800_1849
svediakorp-sec494-SchwerinM_SmasakerAfEn1.conllu
1800_1849
svediakorp-sec254-CederborghF_BerattelseOmJohnHall.conllu
1800_1849
svediakorp-sec987-spf144.conllu
1800_1849
svediakorp-sec988-spf145.conllu
1800_1849
svediakorp-sec545-WallinJO_Vitterhetsforsok.conllu*
1850_1899
svediakorp-sec452-NyblomH_FantasierFyra.conllu
1850_1899
svediakorp-sec486-SchwartzMS_BellmansSkor.conllu
1850_1899
svediakorp-sec1108-spf265.conllu
1850_1899
svediakorp-sec306-FrodingG_RagglerAPaschaserI.conllu**
1850_1899
svediakorp-sec519-StrandbergCG_EsaiasTegner1853.conllu*
1850_1899
svediakorp-sec1102-spf259.conllu
1850_1899
svediakorp-sec307-FrodingG_RagglerAPaschaserII.conllu*
1850_1899
svediakorp-sec208-Anonym_DetGrasligaMordet.conllu
1850_1899
svediakorp-sec339-HanssonO_LiteraraSilhouetter.conllu
1850_1899
svediakorp-sec885-spf42.conllu
1850_1899
svediakorp-sec492-SchwartzMS_PaGothakanal.conllu
1850_1899
svediakorp-sec1031-spf188.conllu
1850_1899
svediakorp-sec415-LevertinO_PepitasBrollop.conllu
1850_1899
svediakorp-sec1063-spf220.conllu
1900_1950
svediakorp-sec631-HasselskogN_HallaHallaGronkoping.conllu
1900_1950
svediakorp-sec1033-spf190.conllu
1900_1950
svediakorp-sec397-AngeredStrandbergH_UnderSodernsSol.conllu
1900_1950
svediakorp-sec609-EngstromA_GenomMinaGuldbagadeGlasogon.conllu
1900_1950
svediakorp-sec613-EngstromA_StrindbergOchJag.conllu
1900_1950
svediakorp-sec802-SoderholmK_MorkretOchManniskan.conllu*
1900_1950
svediakorp-sec639-HeidenstamV_Proletarfilosofiens.conllu
1900_1950
svediakorp-sec1018-spf175.conllu
"""

# Split the lines and clean them
lines = [line.strip() for line in data.splitlines() if line.strip()]

# Create dictionary
period_dict = {}
current_period = None

for line in lines:
    # If the line does not end with '.conllu', it's a period name
    if not line.endswith(".conllu") and not line.endswith(".conllu*") and not line.endswith(".conllu**") and not line.endswith(".conllu***"):
        current_period = line
        if current_period not in period_dict:
            period_dict[current_period] = []
    else:
        # Remove extra '*' from filenames
        clean_file = line.rstrip('*')
        period_dict[current_period].append(clean_file)

# Example output
for period, files in period_dict.items():
    print(period, ":", files)


#Step 3 — Creating a Mapping of Files to Periods

In [ ]:
files_for_iaa = {}
for period, samples in period_dict.items():
    files_for_iaa.setdefault(period, [])
    for sample in samples:
        for k,v in common_files.items():
            if sample in v:
                left, right = k.split("_")
                path1 = extract_path/left/sample
                path2 = extract_path/right/sample
                files_for_iaa[period].append((str(path1), str(path2)))
print(files_for_iaa)



#Step 4 — Creating a Table of IAA Scores

In [ ]:
import pyconll
import numpy as np
from sklearn.metrics import accuracy_score, cohen_kappa_score,precision_score, recall_score, f1_score
import pandas as pd



def safe_kappa(a, b):
    # avoid 0/0 warnings
    if len(set(a)) > 1 and len(set(b)) > 1:
        return cohen_kappa_score(a, b)
    return np.nan

def ck_score(an_one_deprels, an_two_deprels, an_one_heads, an_two_heads,
                      an_one_upos, an_two_upos, an_one_las, an_two_las):

    upos_ks = safe_kappa(an_one_upos, an_two_upos )
    deprel_ks = safe_kappa(an_one_deprels, an_two_deprels)
    heads_ks = safe_kappa(an_one_heads, an_two_heads)

    las_ks = safe_kappa(an_one_las, an_two_las)
    return upos_ks, deprel_ks, heads_ks, las_ks

def agrement(an_one_deprels, an_two_deprels,
             an_one_heads, an_two_heads,
             an_one_upos, an_two_upos,
             arr_one_las, arr_two_las):
    upos_agreement = accuracy_score(an_one_upos, an_two_upos)
    deprel_agreement = accuracy_score(an_one_deprels, an_two_deprels)
    heads_agreement = accuracy_score(an_one_heads, an_two_heads)


    las_pres = precision_score( arr_one_las, arr_two_las, average="micro", zero_division=0)
    #las_rec = recall_score(h_d_pair_long, h_d_pair_short, average="macro")
    #las_f1 =  f1_score(h_d_pair_long, h_d_pair_short, average="macro")

    return upos_agreement, deprel_agreement, heads_agreement, las_pres



def determine_head_id(id, n_tokens_hopped_over):
  """ Function for determining the correct head id of a token
  based on the id from the conllu file and the number of
  tokens hopped over because of merging/splitting"""
  if id is None:
    return "x"
  elif id == "0": # head id is root
    return "0"
  else: # Add number of hopped over tokens to head id
    return str(int(id)+n_tokens_hopped_over)

def features_lists(ann_one, ann_two):
    an_long_upos, an_long_heads, an_long_deprels, h_d_pair_long = [],[],[], []
    an_short_upos, an_short_heads, an_short_deprels, h_d_pair_short = [],[],[],[]
    for s1 in ann_one:
        for s2 in ann_two:
            if s1.id != s2.id:
                pass
            else:
                # Determine which sentence is longer
                if len(s1) > len(s2):
                    longer_sent = s1
                    shorter_sent = s2
                else: # s2 is longer or they are equally long
                    longer_sent = s2
                    shorter_sent = s1
                #print(len(longer_sent),longer_sent[-1].form,len(shorter_sent), shorter_sent[-1].form)
                ind = 0
                n_tokens_hopped_over = 0 # Counter for number of hopped-over tokens in longer sentence - Astrid
                for t_longer in longer_sent:
                    #if shorter_sent[ind]:
                    t_shorter = shorter_sent[ind]
                    # Add the longer sentence's token to lists
                    an_long_deprels.append(t_longer.deprel if t_longer.deprel is not None else "x")
                    an_long_heads.append(t_longer.head if t_longer.head is not None else "x")
                    an_long_upos.append(t_longer.upos if t_longer.upos is not None else "x")
                    if t_longer.form == t_shorter.form: # If tokens have the same form
                            # Add the shorter sentence's token to lists right away
                        an_short_deprels.append(t_shorter.deprel if t_shorter.deprel is not None else "x")
                        an_short_heads.append(determine_head_id(t_shorter.head, n_tokens_hopped_over))
                        an_short_upos.append(t_shorter.upos if t_shorter.upos is not None else "x")
                        ind += 1 # update index to move to next token in the shorter sentence
                    else: # Tokens at index i do not have the same form - a token has been merged or split in one of the sentences
                        an_short_deprels.append("X")
                        an_short_heads.append("X")
                        an_short_upos.append("X")
                        #print(t_shorter.deprel, t_shorter.head, t_shorter.upos, t_shorter.form)
                        if t_shorter.form.endswith(t_longer.form): # We have reached the end of the merged/split expression
                            ind += 1 # continue to the next token in the shorter sentence
                        else:
                            # otherwise, we continue to the longer sentence's next token, but handle the same token again for the shorter sentence
                            n_tokens_hopped_over += 1 # Update counter to indicate that we hopped over a short sent token - Astrid"""
                h_d_pair_long = [f"{h}-{d}" for h,d in zip(an_long_heads, an_long_deprels)] # create 'HEAD_DEPREL pairs for calculating LAS
                h_d_pair_short = [f"{h}-{d}" for h,d in zip(an_short_heads, an_short_deprels)]

    return an_long_deprels, an_long_heads, an_long_upos, an_short_deprels, an_short_heads, an_short_upos, h_d_pair_long, h_d_pair_short



d = {}
for period, files in files_for_iaa.items():
    d.setdefault(period, {})
    for file1, file2 in files:
        pair = file1.split("/")[-2] + "_" + file2.split("/")[-2]
        s = file1.split("/")[-1].split("-")[-2]
        #print(pair, s)
        d[period].setdefault(pair, {})
        annotator_one = pyconll.load_from_file(file1)
        annotator_two = pyconll.load_from_file(file2)
        # Creating feature lists from CONLLU for a pair of annotators
        (an_one_deprels, an_one_heads,
         an_one_upos, an_two_deprels,
         an_two_heads, an_two_upos,
         arr_one_las, arr_two_las) = features_lists(annotator_one, annotator_two)
        # Computing agreement scores
        if len(an_one_deprels) == 0 or len(an_two_deprels) == 0:
            print(f"!!!Shouldn't be here:{pair},{s}")
        else:
            (upos_agreement, deprel_agreement,
            heads_agreement, las_iaa) = agrement(an_one_deprels, an_two_deprels,
                                                an_one_heads, an_two_heads,
                                                    an_one_upos, an_two_upos,
                                                    arr_one_las, arr_two_las)
        #print(f"LAS agreement: {las_iaa :.4f}")

        # Computing Cohen-Kappa scores:
        (upos_ks, deprel_ks,
         heads_ks, las_ks) = ck_score(an_one_deprels, an_two_deprels,
                                               an_one_heads, an_two_heads,
                                               an_one_upos, an_two_upos,
                                               arr_one_las, arr_two_las)


        d[period][pair][s] = (
    f"{upos_agreement:.4f}, {deprel_agreement:.4f}, {heads_agreement:.4f}, {las_iaa:.4f}, "
    f"{upos_ks:.4f}, {deprel_ks:.4f}, {heads_ks:.4f}, {las_ks:.4f}"
)


#print(d)
rows = []
for period, pairs in d.items():
    for pair, secs in pairs.items():
        for sec, scores in secs.items():
            upos, deprel, heads, las, upos_ks, dl_ks, h_ks, las_ks = map(float, scores.split(", "))
            rows.append({
                'Period': period,
                'Pair': pair,
                'Sample': sec,
                'UPOS': 100 * upos,
                'Deprel': 100 * deprel,
                'UAS': 100 * heads,
                'LAS': 100* las,
                'Cohen-Kappa UPOS': 100 * upos_ks,
                'Cohen-Kappa DEPREL': 100 * dl_ks,
                'Cohen-Kappa UAS': 100 * h_ks,
                'Cohen-Kappa LAS': 100 * las_ks
            })

# Create DataFrame
df = pd.DataFrame(rows)
print("Inter-Annotator Agreement per Period and Sample:")
df

In [ ]:
mean_scores_by_period = df.groupby('Period')[['UPOS', 'Deprel', 'UAS', 'LAS',
                                              'Cohen-Kappa UPOS','Cohen-Kappa DEPREL',
                                              'Cohen-Kappa UAS','Cohen-Kappa LAS' ]].mean()
print("IAA Mean Scores by Period:")
display(mean_scores_by_period)

#Testing IAA on Two Samples in Independent Notebook Cells


In [ ]:
import pyconll
an_one = "/content/extracted_annotations/dump_DigPhil_Diachronic_Treebanks_clean/lingfilalanev/svediakorp-sec277-EnbomPU_MedborgeligtSkalde.conllu"
an_two = "/content/extracted_annotations/dump_DigPhil_Diachronic_Treebanks_clean/astbi/svediakorp-sec277-EnbomPU_MedborgeligtSkalde.conllu"
train_one = pyconll.load_from_file(an_one)
train_two = pyconll.load_from_file(an_two)
print(len(train_one), len(train_two))


In [ ]:
"""This cell allows you to select a specific sentence
from the dataset for inspection or experimentation."""



train_one= train_one[1:2]   # use the range to select the particular sentence
train_two = train_two[1:2]
print(len(train_one), len(train_two))
def one_sent(train, path):
    with open(path, "w", encoding="utf-8") as f:
        for s in train:
            print("TOKENS:", [t.form for t in s])
            f.write(s.conll())
            f.write("\n\n")

one_sent(train_one, "one.conllu")
one_sent(train_two, "two.conllu")



In [ ]:
#from re import X
import numpy as np

def determine_head_id(id, n_tokens_hopped_over):
  """ Function for determining the correct head id of a token
  based on the id from the conllu file and the number of
  tokens hopped over because of merging/splitting"""
  if id is None:
    return "x"
  elif id == "0": # head id is root
    return "0"
  else: # Add number of hopped over tokens to head id
    return str(int(id)+n_tokens_hopped_over)

an_long_upos, an_long_heads, an_long_deprels = [],[],[]
an_short_upos, an_short_heads, an_short_deprels = [],[],[]
for s1 in train_one:
    for s2 in train_two:
        if s1.id == s2.id:
            # Determine which sentence is longer
            if len(s1) > len(s2):
                longer_sent = s1
                shorter_sent = s2
            else: # s2 is longer or they are equally long
                longer_sent = s2
                shorter_sent = s1
            ind = 0
            n_tokens_hopped_over = 0 # Counter for number of hopped-over tokens in longer sentence - Astrid
            for t_longer in longer_sent:
              t_shorter = shorter_sent[ind]
              # Add the longer sentence's token to lists
              an_long_deprels.append(t_longer.deprel if t_longer.deprel is not None else "x")
              an_long_heads.append(t_longer.head if t_longer.head is not None else "x")
              an_long_upos.append(t_longer.upos if t_longer.upos is not None else "x")
              if t_longer.form == t_shorter.form: # If tokens have the same form
                # Add the shorter sentence's token to lists right away
                an_short_deprels.append(t_shorter.deprel if t_shorter.deprel is not None else "x")
                an_short_heads.append(determine_head_id(t_shorter.head, n_tokens_hopped_over))
                an_short_upos.append(t_shorter.upos if t_shorter.upos is not None else "x")
                ind += 1 # update index to move to next token in the shorter sentence
              else: # Tokens at index i do not have the same form - a token has been merged or split in one of the sentences
                an_short_deprels.append("X")
                an_short_heads.append("X")
                an_short_upos.append("X")
                print(t_shorter.deprel, t_shorter.head, t_shorter.upos, t_shorter.form)
                if t_shorter.form.endswith(t_longer.form): # We have reached the end of the merged/split expression
                  ind += 1 # continue to the next token in the shorter sentence
                else:
                  # otherwise, we continue to the longer sentence's next token, but handle the same token again for the shorter sentence
                  n_tokens_hopped_over += 1 # Update counter to indicate that we hopped over a short sent token - Astrid"""

h_d_pair_long = [f"{h}-{d}" for h,d in zip(an_long_heads, an_long_deprels)] # create 'HEAD_DEPREL pairs for calculating LAS
h_d_pair_short = [f"{h}-{d}" for h,d in zip(an_short_heads, an_short_deprels)]


#print(an_long_upos)
#print(an_short_upos)
#print(an_long_heads)
#print(an_short_heads)
#print(an_long_deprels)
#print(an_short_deprels)
#print(h_d_pair_long)
#print(h_d_pair_short)

# TODO Should we have some tag/head/deprel instead of X in the short sentence? Which one? What if one of the tokens in the split/merged expression is ROOT?

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import numpy as np



upos_agreement = accuracy_score(an_long_upos, an_short_upos)
deprel_agreement = accuracy_score(an_long_deprels, an_short_deprels)
heads_agreement = accuracy_score(an_long_heads, an_short_heads)

upos_ag_prec = precision_score(an_short_upos,an_long_upos, average='weighted', zero_division=0)
deprel_ag_prec = precision_score(an_long_deprels, an_short_deprels, average='weighted', zero_division=0)
heads_ag_prec = precision_score(an_long_heads, an_short_heads, average='weighted', zero_division=0)

upos_ag_rec = recall_score(an_long_upos, an_short_upos, average='weighted', zero_division=0)
deprel_ag_rec = recall_score(an_long_deprels, an_short_deprels, average='weighted', zero_division=0)
heads_ag_rec = recall_score(an_long_heads, an_short_heads, average='weighted', zero_division=0)

upos_ag_f1 = f1_score(an_long_upos, an_short_upos, average='weighted', zero_division=0)
deprel_ag_f1 = f1_score(an_long_deprels, an_short_deprels, average='weighted', zero_division=0)
heads_ag_f1 = f1_score(an_long_heads, an_short_heads, average='weighted', zero_division=0)



las_pres = precision_score(h_d_pair_long, h_d_pair_short, average="macro")
las_rec = recall_score(h_d_pair_long, h_d_pair_short, average="macro")
las_f1 =  f1_score(h_d_pair_long, h_d_pair_short, average="macro")

sc_tab = pd.DataFrame({
    "Agreement": [upos_agreement, deprel_agreement, heads_agreement, "-"],
    "Precision": [upos_ag_prec, deprel_ag_prec, heads_ag_prec, las_pres],
    "Recall": [upos_ag_rec, deprel_ag_rec, heads_ag_rec, las_rec],
    "F1": [upos_ag_f1, deprel_ag_f1, heads_ag_f1, las_f1]
}, index=["UPOS", "DEPREL", "UAS", "LAS"])
sc_tab

In [ ]:
!python conll18_ud_eval.py -v "two.conllu" "one.conllu"

In [ ]:
!python conll18_ud_eval.py -c "two.conllu" "one.conllu"

In [ ]:
!python conll18_ud_eval.py -v "one.conllu" "two.conllu"

In [ ]:
!python conll18_ud_eval.py -c "one.conllu" "two.conllu"

In [ ]:
from sklearn.metrics import cohen_kappa_score

upos_ks = cohen_kappa_score(an_long_upos, an_short_upos )
print(f'Cohen-kappa score for UPOS: {upos_ks :.4f}')
deprel_ks = cohen_kappa_score(an_long_deprels, an_short_deprels)
print(f'Cohen-kappa score for DEPREL: {deprel_ks :.4f}')
heads_ks = cohen_kappa_score(an_long_heads, an_short_heads)
print(f'Cohen-kappa score for HEADS: {heads_ks:.4f}')


ann_one_lbl = np.array(['_'.join(x) for x in arr_one_las.astype(str)])
ann_two_lbl = np.array(['_'.join(x) for x in arr_two_las.astype(str)])

kappa = cohen_kappa_score(ann_one_lbl, ann_two_lbl)
print(f'Cohen-kappa score for LAS: {kappa :.4f}')